In [3]:
import pandas as pd

#import the dataset
df = pd.read_csv('./dataset/synthetic_logs.csv')
# df.head()
df

,timestamp,source,log_message,target_label,complexity
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert
...,...,...,...,...,...
2405,13-08-2025 07:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert
2406,01-11-2025 05:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert
2407,03-08-2025 03:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert
2408,11-11-2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert


In [2]:
#identify unique source
unique_sources = df['source'].unique()
print(unique_sources)
#identify unique target_labels
unique_target_labels = df['target_label'].unique()
unique_target_labels

['ModernCRM' 'AnalyticsEngine' 'ModernHR' 'BillingSystem' 'ThirdPartyAPI'
 'LegacyCRM']


array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [3]:
# Import necessary libraries
from sentence_transformers import SentenceTransformer

print("Loading SentenceTransformer model (all-MiniLM-L6-v2)...")
# Load a pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")

# Extract log messages and generate embeddings
# log_messages = df['log_message'].values
embeddings = model.encode(df['log_message'].tolist(), show_progress_bar=True)

C:\Users\kumar\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading SentenceTransformer model (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5943.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


Batches: 100%|██████████| 76/76 [00:16<00:00,  4.56it/s]


In [4]:
from sklearn.cluster import DBSCAN

# Apply DBSCAN clustering
# Parameters:
# eps: maximum distance between two samples for them to be considered in the same neighborhood (0.5 is a good starting point)
# min_samples: minimum number of samples in a neighborhood for a point to be considered core point (default 5)
# eps and min_samples can be adjusted based on the dataset and desired cluster granularity (noise reduction)
dbscan = DBSCAN(eps=0.8, min_samples=5, metric='euclidean')
clusters = dbscan.fit_predict(embeddings)

# Add cluster labels to dataframe
df['cluster'] = clusters
print("\nCluster distribution:")
df['cluster'].head()
df.head()
# print(df['cluster'].value_counts().sort_index())


Cluster distribution:


,timestamp,source,log_message,target_label,complexity,cluster
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [5]:
#Identify lagest cluster with similar patterns
clusters_counts = df['cluster'].value_counts()
largest_cluster = clusters_counts[clusters_counts > 10].index

largest_cluster  

for clusters in largest_cluster:
    print(f"\nCluster {clusters}:")
    print(df[df['cluster'] == clusters]['log_message'].head(5).to_string(index=False))



Cluster 0:
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...

Cluster 7:
Multiple bad login attempts detected on user 85...
Alert: brute force login attempt from 192.168.8...
Suspicious login activity detected from 192.168...
Denied access attempt on restricted account Acc...
Abnormal system behavior on server 40, potentia...

Cluster 9:
Account with ID 5351 created by User634.
                User User685 logged out.
System reboot initiated by user User243.
                 User User395 logged in.
                 User User225 logged in.

Cluster 5:
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.resource_tracker [req-addc1839-2ed...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd

In [6]:
#Regex Classification
import re

def regex_classify_log(log_message):
    # Define regex patterns for different log types
    regex_patterns = {
        'System Notification': [
            r'(File data_\d+.csv uploaded successfully by use...)',
            r'(Backup completed successfully.)',
            r'(System updated to version [\d\.]+)',
            r'(System reboot initiated by user User\d+.)',
            r'(\s*Backup (started|ended) at [\d\s\-\:]*)',
            r'(Disk cleanup completed successfully.)'
        ],
        'User Action': [
            r'(Account with ID \d+ created by User\d+.)',
            r'(\s*User\s+User\d+\s+logged\s(in|out))'
        ]
    }

    # Check log message against regex patterns and identify log type
    for label, patterns in regex_patterns.items():
        for pattern in patterns:
            if re.search(pattern, log_message, re.IGNORECASE):
                return label
    return None #'No regex pattern matched'

In [7]:
# Add regex labels to log messages
df['regex_label'] = df['log_message'].apply(regex_classify_log)
# df[['log_message', 'regex_label']].head(20)

# df[df.regex_label != 'No regex pattern matched'][['log_message', 'regex_label']].head(20)
df[df.regex_label.notna()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
7,10-11-2025 08:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,regex,4,System Notification
14,01-04-2025 01:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,regex,4,System Notification
15,05-01-2025 09:41,ModernCRM,Backup completed successfully.,System Notification,regex,8,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,regex,9,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,regex,9,User Action
...,...,...,...,...,...,...,...
2376,6/27/2025 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,regex,13,System Notification
2381,09-05-2025 06:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,regex,16,System Notification
2394,04-03-2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,regex,16,System Notification
2395,05-02-2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,regex,10,System Notification


In [8]:
# Filter non-regex logs with target_label <= 5
# Identify sources and target_labels with low frequency
non_regex_df = df[df['regex_label'].isna()]
non_regex_df_ls = ( 
    non_regex_df
    .groupby(['source','target_label'])
    .size()
    .loc[lambda count: count <= 5]
    .reset_index(name='count')
    .sort_values(['target_label', 'count'], ascending=[True, False])
)

non_regex_df_ls

,source,target_label,count
0,LegacyCRM,Deprecation Warning,3
1,LegacyCRM,Workflow Error,4


In [9]:
# Filtered out low frequency sources which can't be handled by BERT classification
bert_df= non_regex_df[non_regex_df.source != 'LegacyCRM']
bert_df.source.unique().tolist()


['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']

In [10]:
# Generate embeddings 
filtered_embeddings = model.encode(bert_df['log_message'].tolist(), show_progress_bar=True)
filtered_embeddings[:5]

Batches: 100%|██████████| 60/60 [00:18<00:00,  3.32it/s]


array([[-0.10293962,  0.03354593, -0.02202606, ...,  0.00457791,
        -0.04259717,  0.00322621],
       [ 0.00804573, -0.03573923,  0.04938737, ...,  0.01538319,
        -0.06230948, -0.02774663],
       [-0.00908223,  0.13003924, -0.05275566, ...,  0.02014104,
        -0.05117098, -0.02930296],
       [-0.09751043,  0.04911299, -0.03977422, ...,  0.024775  ,
        -0.03546078, -0.000186  ],
       [-0.10468339,  0.05926036, -0.02488496, ...,  0.02502052,
        -0.03719298, -0.02568911]], shape=(5, 384), dtype=float32)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

x = filtered_embeddings
y = bert_df['target_label']

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42,
    stratify=bert_df['target_label']
)

clf = LogisticRegression(max_iter=1000, solver='lbfgs')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("Train accuracy:", accuracy_score(y_train, clf.predict(X_train)))
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

Train samples: 1332
Test samples: 571
Train accuracy: 0.996996996996997
Test accuracy: 0.9982486865148862

Classification report:
                 precision    recall  f1-score   support

Critical Error       1.00      0.98      0.99        48
         Error       1.00      1.00      1.00        53
   HTTP Status       1.00      1.00      1.00       305
Resource Usage       1.00      1.00      1.00        53
Security Alert       0.99      1.00      1.00       112

      accuracy                           1.00       571
     macro avg       1.00      1.00      1.00       571
  weighted avg       1.00      1.00      1.00       571



In [18]:
import joblib

# Save the trained model to a file
joblib.dump(clf, '../models/log_classifier.joblib')

['../models/log_classifier.joblib']